# Assignment 3 – Retrieval-Augmented Generation (RAG)

**Goal:** Build a RAG system over the original BBC News article text files, then compare chunk sizes, retrieval methods, and re-ranking methods as required by the assignment.

**Important:** This assignment does not require model training or numerical evaluation metrics. I inspect retrieved chunks and generated answers using examples.

## Before running
1. Download and extract the BBC News Summary Kaggle dataset.
2. Make sure the original `.txt` files are inside the `News Articles` folder.
3. Update `DATASET_DIR` in the next section if your folder is in a different location.
4. Install and start Ollama on your computer, then download the required model with `ollama pull mistral`.

## 1. Install the required libraries

Run this cell once. If Jupyter asks for a kernel restart after installation, restart the kernel and continue from the next section.

In [30]:
%pip install -q sentence-transformers transformers langchain-text-splitters faiss-cpu rank-bm25 pandas matplotlib requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and dataset path

The notebook searches inside `DATASET_DIR` for the `News Articles` folder. Keep the original dataset structure after extraction.

In [31]:
from pathlib import Path
import re
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import faiss
from rank_bm25 import BM25Okapi
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Notebook is expected to be saved inside Assignment-3.
# Change this path if your extracted dataset is stored elsewhere.
DATASET_DIR = Path(r"D:\SCMS\Python Projects\Assignment-3\BBC News Summary")

def find_news_articles_folder(root: Path):
    if not root.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {root.resolve()}\n"
            "Update DATASET_DIR to the folder where you extracted the BBC dataset."
        )
    matches = [p for p in root.rglob("*") if p.is_dir() and p.name.lower() == "news articles"]
    if matches:
        return matches[0]

    # Fallback: find a folder containing many .txt files in nested folders.
    txt_files = list(root.rglob("*.txt"))
    if txt_files:
        return root

    raise FileNotFoundError(
        f"No 'News Articles' folder or .txt files found under {root.resolve()}."
    )

NEWS_DIR = find_news_articles_folder(DATASET_DIR)
print("Using article folder:", NEWS_DIR.resolve())

Using article folder: D:\SCMS\Python Projects\Assignment-3\BBC News Summary\News Articles


## 3. Load the original BBC news articles

I load `.txt` files only from the original **News Articles** folder, not the Summaries folder. I keep each article's filename and category (when the category is represented by a parent folder).

In [32]:
documents = []

for file_path in NEWS_DIR.rglob("*.txt"):
    try:
        text = file_path.read_text(encoding="utf-8", errors="ignore").strip()
    except Exception as e:
        print(f"Skipped {file_path.name}: {e}")
        continue

    if text:
        documents.append({
            "text": text,
            "source": file_path.name,
            "category": file_path.parent.name
        })

print("Articles loaded:", len(documents))
print("\nExample:")
print("Filename:", documents[0]["source"] if documents else "No files found")
print((documents[0]["text"][:500] if documents else "Check DATASET_DIR and NEWS_DIR."))

Articles loaded: 2225

Example:
Filename: 001.txt
Ad sales boost Time Warner profit

Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.

The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.

Time W


## 4. Recursive text splitting: 100, 150 and 200 tokens

The assignment asks for recursive splitting, chunk sizes of 100, 150 and 200 tokens, and an overlap of 20 tokens.

I use the tokenizer associated with MiniLM to count tokens. The recursive splitter uses those token counts as its chunk-length function.

In [33]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

def token_count(text):
    return len(tokenizer.encode(text, add_special_tokens=False))

def make_chunks(chunk_size, overlap=20):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        length_function=token_count,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = []
    for doc in documents:
        parts = splitter.split_text(doc["text"])
        for part_no, part in enumerate(parts):
            if part.strip():
                chunks.append({
                    "text": part.strip(),
                    "source": doc["source"],
                    "category": doc["category"],
                    "chunk_id": part_no
                })
    return chunks

chunks_by_size = {}
chunk_counts = []

for size in [100, 150, 200]:
    chunks_by_size[size] = make_chunks(size, overlap=20)
    chunk_counts.append({
        "Chunk size (tokens)": size,
        "Overlap (tokens)": 20,
        "Number of chunks": len(chunks_by_size[size])
    })

chunk_count_table = pd.DataFrame(chunk_counts)
display(chunk_count_table)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (557 > 512). Running this sequence through the model will result in indexing errors


,Chunk size (tokens),Overlap (tokens),Number of chunks
0,100,20,16629
1,150,20,11102
2,200,20,8108


## 5. Create embeddings and FAISS HNSW index

Required embedding model: `sentence-transformers/all-MiniLM-L6-v2`.

I use a FAISS HNSW index. I normalize the embeddings, so inner-product search acts like cosine-similarity search.

In [34]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def build_faiss_hnsw(chunks):
    texts = [chunk["text"] for chunk in chunks]
    vectors = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    ).astype("float32")

    dimension = vectors.shape[1]
    index = faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)
    index.hnsw.efConstruction = 80
    index.hnsw.efSearch = 64
    index.add(vectors)
    return index, vectors

# Build an index for each required chunk size so chunking can be compared fairly.
faiss_data = {}
for size in [100, 150, 200]:
    print(f"Building FAISS HNSW index for {size}-token chunks...")
    index, vectors = build_faiss_hnsw(chunks_by_size[size])
    faiss_data[size] = {"index": index, "vectors": vectors}

print("FAISS HNSW indexes are ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2534.23it/s]


Building FAISS HNSW index for 100-token chunks...


Batches: 100%|██████████| 520/520 [03:14<00:00,  2.67it/s]


Building FAISS HNSW index for 150-token chunks...


Batches: 100%|██████████| 347/347 [03:05<00:00,  1.87it/s]


Building FAISS HNSW index for 200-token chunks...


Batches: 100%|██████████| 254/254 [03:11<00:00,  1.33it/s]


FAISS HNSW indexes are ready.


## 6. Build BM25 indexes

BM25 is keyword-based retrieval. I tokenize each chunk into lowercase words and build one BM25 index for each chunk size.

In [35]:
def simple_tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

bm25_data = {}
for size in [100, 150, 200]:
    tokenized_chunks = [simple_tokenize(c["text"]) for c in chunks_by_size[size]]
    bm25_data[size] = BM25Okapi(tokenized_chunks)

print("BM25 indexes are ready.")

BM25 indexes are ready.


## 7. Retrieval methods: BM25, Dense and Hybrid

Each retrieval method returns the **top 5 chunks**, as required.

Hybrid retrieval uses Reciprocal Rank Fusion (RRF) to combine BM25 and dense rankings. This combines the rankings without needing to compare BM25 scores directly with cosine-similarity scores.

In [ ]:
def bm25_search(question, size=150, top_k=5):
    scores = bm25_data[size].get_scores(simple_tokenize(question))
    order = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in order]

def dense_search(question, size=150, top_k=5):
    query_vector = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    ).astype("float32")
    scores, indices = faiss_data[size]["index"].search(query_vector, top_k)
    return [(int(i), float(score)) for i, score in zip(indices[0], scores[0]) if i >= 0]

def hybrid_search(question, size=150, top_k=5, candidate_k=20):
    bm25_results = bm25_search(question, size=size, top_k=candidate_k)
    dense_results = dense_search(question, size=size, top_k=candidate_k)

    rrf_scores = {}
    rrf_constant = 60

    for rank, (idx, _) in enumerate(bm25_results, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_constant + rank)

    for rank, (idx, _) in enumerate(dense_results, start=1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1 / (rrf_constant + rank)

    ranked = sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)
    return [(idx, score) for idx, score in ranked[:top_k]]

def show_results(results, chunks):
    rows = []
    for rank, (idx, score) in enumerate(results, start=1):
        chunk = chunks[idx]
        rows.append({
            "Rank": rank,
            "Filename": chunk["source"],
            "Category": chunk["category"],
            "Score": round(score, 4),
            "Chunk text": chunk["text"][:500]
        })
    return pd.DataFrame(rows)

# Quick test with one question grounded in the article collection.
test_question = "How did TimeWarner's quarterly profit compare with the previous year's figure, and what was the latest amount?"
display(show_results(hybrid_search(test_question, size=150), chunks_by_size[150]))

## 8. Re-ranking: Bi-Encoder and Cross-Encoder

For a fair comparison, both re-rankers receive the **same candidate set**: the top 20 chunks returned by hybrid retrieval. I compare the original hybrid top five with the top five selected after each reranker.

- **Bi-Encoder:** embeds the question and candidate chunks with MiniLM, then sorts by cosine similarity.
- **Cross-Encoder:** scores each question-and-chunk pair using `cross-encoder/ms-marco-MiniLM-L-6-v2`.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank_biencoder(question, candidate_results, chunks, top_k=5):
    candidate_indices = [idx for idx, _ in candidate_results]
    candidate_texts = [chunks[idx]["text"] for idx in candidate_indices]

    q_vec = embedding_model.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    d_vecs = embedding_model.encode(
        candidate_texts, convert_to_numpy=True, normalize_embeddings=True
    )

    scores = np.dot(d_vecs, q_vec)
    ranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)
    return [(int(idx), float(score)) for idx, score in ranked[:top_k]]

def rerank_crossencoder(question, candidate_results, chunks, top_k=5):
    candidate_indices = [idx for idx, _ in candidate_results]
    pairs = [(question, chunks[idx]["text"]) for idx in candidate_indices]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(candidate_indices, scores), key=lambda x: x[1], reverse=True)
    return [(int(idx), float(score)) for idx, score in ranked[:top_k]]

# Example: all three lists below use the same hybrid candidate set.
hybrid_candidates = hybrid_search(
    test_question, size=150, top_k=20, candidate_k=20
)
bi_results = rerank_biencoder(test_question, hybrid_candidates, chunks_by_size[150])
cross_results = rerank_crossencoder(test_question, hybrid_candidates, chunks_by_size[150])

print("Hybrid order:")
display(show_results(hybrid_candidates, chunks_by_size[150]))
print("Bi-Encoder order:")
display(show_results(bi_results, chunks_by_size[150]))
print("Cross-Encoder order:")
display(show_results(cross_results, chunks_by_size[150]))

## 9. Answer generation using Ollama and Mistral

This section follows the assignment requirement to use Ollama with the `mistral` model. Ollama must be running on this computer, and `mistral` must already be downloaded.

In a separate terminal, run:
```bash
ollama pull mistral
ollama serve
```
If Ollama is already running in the background, you only need the pull command.

**Local connection note:** This notebook calls Ollama at `http://localhost:11434`. This works when Jupyter is running on the same computer as Ollama.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

def generate_answer(question, retrieved_results, chunks):
    context_parts = []
    source_names = []

    for rank, (idx, score) in enumerate(retrieved_results, start=1):
        chunk = chunks[idx]
        context_parts.append(
            f"[Source {rank}: {chunk['source']}]\n{chunk['text']}"
        )
        source_names.append(chunk["source"])

    context = "\n\n".join(context_parts)
    prompt = f'''
You answer questions using only the news article context below.
If the answer is not present in the context, say:
"I cannot find this information in the provided articles."
Do not use outside knowledge. Be concise and factual.

Context:
{context}

Question: {question}

Answer:
'''

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False
        },
        timeout=180
    )
    response.raise_for_status()
    answer = response.json()["response"].strip()

    return {
        "question": question,
        "answer": answer,
        "sources": sorted(set(source_names))
    }

# Test answer generation with hybrid retrieval.
answer_result = generate_answer(
    test_question,
    hybrid_search(test_question, size=150, top_k=5),
    chunks_by_size[150]
)
print("Question:", answer_result["question"])
print("\nAnswer:", answer_result["answer"])
print("\nSource files:", ", ".join(answer_result["sources"]))

## 10. Prepare at least 10 questions and compare retrieval methods

The ten questions below are grounded in specific BBC business articles. Their expected source articles are listed in `RAG_Article_Questions.md`; retrieval and answer results are saved under `outputs/article_question_*.csv`.

For every question, I compare BM25, Dense and Hybrid retrieval. I also save the retrieved filenames and text so I can inspect relevance.

In [ ]:
questions = [
    "How did TimeWarner's quarterly profit compare with the previous year's figure, and what was the latest amount?",
    "What pre-tax profit did WPP report, and by what percentage had it increased?",
    "By what average percentage did Marks & Spencer reduce prices in London and the regions?",
    "What actions might the UK government take to avoid breaking its golden rule?",
    "By how much did the IEA revise its 2005 oil-demand growth estimate, and what level of demand did it project?",
    "What annual output was planned for Hyundai's second Indian plant, and what total capacity would that bring?",
    "What happened to the Ifo business-confidence index between January and February?",
    "How much of Colombia's subsidy funding was reported to benefit the richest part of the population?",
    "What were Cadbury Schweppes' underlying pre-tax profits in 2004, and how did currency movements affect the reported increase?",
    "Which two companies agreed to collaborate on flat-screen television technology?"
]

def compare_retrieval_methods(question, size=150):
    methods = {
        "BM25": bm25_search(question, size=size, top_k=5),
        "Dense": dense_search(question, size=size, top_k=5),
        "Hybrid": hybrid_search(question, size=size, top_k=5)
    }

    rows = []
    for method, results in methods.items():
        for rank, (idx, score) in enumerate(results, start=1):
            rows.append({
                "Question": question,
                "Method": method,
                "Rank": rank,
                "Filename": chunks_by_size[size][idx]["source"],
                "Score": round(score, 4),
                "Chunk text": chunks_by_size[size][idx]["text"][:350]
            })
    return pd.DataFrame(rows)

retrieval_comparisons = []
for q_no, question in enumerate(questions, start=1):
    print(f"Question {q_no}: {question}")
    comparison = compare_retrieval_methods(question, size=150)
    display(comparison)
    retrieval_comparisons.append(comparison)

In [ ]:
specific_question = questions[0]

retrieved = hybrid_search(
    specific_question,
    size=150,
    top_k=5
)

display(show_results(retrieved, chunks_by_size[150]))

result = generate_answer(
    specific_question,
    retrieved,
    chunks_by_size[150]
)

print("Question:", result["question"])
print("\nAnswer:", result["answer"])
print("\nSources:", result["sources"])

## 11. Compare chunk sizes: 100 vs 150 vs 200 tokens

I compare the top retrieved chunks for the same question using hybrid retrieval and inspect whether each result contains enough information to answer the question.

In [ ]:
chunk_size_comparison = []

for question in questions[:5]:
    for size in [100, 150, 200]:
        results = hybrid_search(question, size=size, top_k=5)
        chunk_size_comparison.append({
            "Question": question,
            "Chunk size": size,
            "Retrieved filenames": ", ".join(chunks_by_size[size][idx]["source"] for idx, _ in results),
            "Top chunk preview": chunks_by_size[size][results[0][0]]["text"][:300] if results else ""
        })

chunk_size_comparison_df = pd.DataFrame(chunk_size_comparison)
display(chunk_size_comparison_df)

## 12. Compare re-ranking and generated answers

For each selected question, I compare:
1. Hybrid retrieval without re-ranking.
2. Bi-Encoder re-ranking.
3. Cross-Encoder re-ranking.

For each question, I use the same top-20 hybrid candidate set for both rerankers. I compare the hybrid top five without reranking against the top five returned by each reranker.

In [ ]:
reranking_observations = []

for question in questions[:5]:
    candidates = hybrid_search(
        question, size=150, top_k=20, candidate_k=20
    )
    hybrid_top_five = candidates[:5]
    bi_ranked = rerank_biencoder(
        question, candidates, chunks_by_size[150], top_k=5
    )
    cross_ranked = rerank_crossencoder(
        question, candidates, chunks_by_size[150], top_k=5
    )

    for method, results in [
        ("Hybrid - no reranking", hybrid_top_five),
        ("Bi-Encoder", bi_ranked),
        ("Cross-Encoder", cross_ranked)
    ]:
        result = generate_answer(question, results, chunks_by_size[150])
        reranking_observations.append({
            "Question": question,
            "Method": method,
            "Answer": result["answer"],
            "Source filenames": ", ".join(result["sources"])
        })

reranking_df = pd.DataFrame(reranking_observations)
display(reranking_df)
reranking_output_path = Path("outputs") / "reranking_answer_comparison.csv"
reranking_output_path.parent.mkdir(parents=True, exist_ok=True)
reranking_df.to_csv(reranking_output_path, index=False)
print("Saved reranking answer comparison to:", reranking_output_path.resolve())

## 13. Required sample questions, answers and source filenames

I include at least five sample questions, generated answers and source filenames. The table below uses the measured article-grounded answers and their Cross-Encoder-reranked hybrid sources.

In [ ]:
sample_answers_df = pd.read_csv(
    Path("outputs") / "article_question_answers.csv", encoding="utf-8-sig"
)
display(sample_answers_df[["Question", "Generated answer", "Source paths"]].head(5))

## 14. Export comparison tables

These CSV files can be used in the written report. They are saved in the same folder as this notebook.

In [47]:
OUTPUT_DIR = Path(".")
chunk_count_table.to_csv(OUTPUT_DIR / "chunk_count_comparison.csv", index=False)
chunk_size_comparison_df.to_csv(OUTPUT_DIR / "chunk_size_comparison.csv", index=False)
reranking_df.to_csv(OUTPUT_DIR / "reranking_comparison.csv", index=False)
sample_answers_df.to_csv(OUTPUT_DIR / "sample_questions_answers.csv", index=False)

if retrieval_comparisons:
    pd.concat(retrieval_comparisons, ignore_index=True).to_csv(
        OUTPUT_DIR / "retrieval_methods_comparison.csv", index=False
    )

print("Comparison CSV files saved in:", OUTPUT_DIR.resolve())

Comparison CSV files saved in: D:\SCMS\Python Projects\Assignment-3


In [45]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral",
        "prompt": "Reply with just the word OK.",
        "stream": False
    },
    timeout=180
)

print("Status code:", response.status_code)
print("Ollama response:")
print(response.text)

Status code: 200
Ollama response:
{"model":"mistral","created_at":"2026-11-14T21:27:24.2131268Z","response":" OK.","done":true,"done_reason":"stop","context":[3,29473,4125,1114,1163,1544,1040,2475,12974,29491,4,29473,12974,29491],"total_duration":1480089100,"load_duration":4706000,"prompt_eval_count":12,"prompt_eval_cached_count":2,"prompt_eval_duration":976412000,"eval_count":3,"eval_duration":460855000}


## 15. My answers to the assignment questions

### Q1 — Document loading and chunking
I loaded all 2,225 original BBC News Articles text files and used recursive token-based splitting with 20-token overlap. I generated 16,629 chunks at size 100, 11,102 at 150, and 8,108 at 200. The 150-token setting looked better in one broad-query example, but I did not establish a generally best size.

### Q2 — Embeddings and vector database
I used `sentence-transformers/all-MiniLM-L6-v2` to embed the chunks and stored the normalized vectors in a FAISS HNSW index. I used inner-product search, which corresponds to cosine similarity for normalized vectors.

### Q3 — Retrieval strategies
I implemented BM25, dense FAISS retrieval, and hybrid Reciprocal Rank Fusion, returning five chunks per method. In my ten article-specific questions, each method retrieved the expected article in its top five for all ten questions. The methods did not return identical chunks: BM25 and Dense shared an average of 2.0 chunks out of five per question, BM25 and Hybrid 3.3, and Dense and Hybrid 3.2. This targeted test does not establish a universal best method.

### Q4 — Re-ranking
I applied both the MiniLM Bi-Encoder and the Cross-Encoder to the same top-20 hybrid candidate set and kept the top five after reranking. Each reranker changed the top-five list for all ten questions compared with no reranking, while the expected article stayed at rank one in every variant.

### Q5 — Answer generation
I used local Ollama with Mistral to answer from the retrieved context and included source filenames. The generated answers correctly returned several requested facts, but I also observed omissions in the TimeWarner and Cadbury answers and a misspelling of Matsushita. The prompt instructs Mistral to say when the context does not contain the answer.

### Q6 — Experiments and observations
I tested ten questions for chunk size, all three retrieval methods, and reranker ordering. I generated ten Mistral answers from Cross-Encoder-ranked results and compared answer generation across all three ranking choices in a separate broad-query experiment. Reranking changed order but did not improve the rank of the expected article in the ten-question test. In the separate broad sports question, the Cross-Encoder and no-reranking answers discussed the betting audit-trail report, while the Bi-Encoder answer gave unrelated athletics details. I therefore found a benefit in that example, but not a consistent improvement across experiments. I consider hybrid retrieval followed by Cross-Encoder reranking a reasonable working combination, not a proven overall winner. I saved the question answers, ranked chunks, and source checks in `outputs/article_question_*.csv`.